In [5]:
#Movie Budget Classification with LLM Features
# Install
!pip install -q llm-feature-gen scikit-learn pandas
import json, shutil, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from llm_feature_gen.discover import discover_features_from_texts
from llm_feature_gen.generate import generate_features_from_texts
from llm_feature_gen.providers.local_provider import LocalProvider

# 1. Load data

df = pd.read_csv("movies.csv")
df = df[df['budget'] > 0]

# Binary classification: high budget (>$150M) vs low budget
median_budget = 150000000
df['label'] = (df['budget'] > median_budget).map({True: 'high', False: 'low'})

# Create text description
def make_text(row):
    return f"Title: {row['title']}. Genres: {row['genres']}. Overview: {str(row['overview'])[:200]}. Director: {row['director']}"

df['text'] = df.apply(make_text, axis=1)

# Take a balanced sample (20 high, 20 low)
high = df[df['label'] == 'high'].head(20)
low = df[df['label'] == 'low'].head(20)
df = pd.concat([high, low])

# Split
train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
print(f"Train: {len(train)} | Test: {len(test)}")


# 2. Create folders:
discover = Path("discover")
train_dir = Path("train")
test_dir = Path("test")
out = Path("outputs")

for d in [discover, train_dir, test_dir, out]:
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir()

for c in ['high', 'low']:
    (train_dir / c).mkdir()
    (test_dir / c).mkdir()

# Discovery files (2 per class)
for c in ['high', 'low']:
    for i, row in train[train['label'] == c].head(2).iterrows():
        (discover / f"{c}_{i}.txt").write_text(row['text'])

# Train/test files
for i, row in train.iterrows():
    (train_dir / row['label'] / f"{i}.txt").write_text(row['text'])
for i, row in test.iterrows():
    (test_dir / row['label'] / f"{i}.txt").write_text(row['text'])

# 3. Discover features with LLM
provider = LocalProvider(
    base_url="https://llm.vse.cz/ollama/v1/",
    api_key="ollama",
    default_text_model="qwen2.5:latest",
    temperature=0.2,
)

features = discover_features_from_texts(
    texts_or_file=discover,
    provider=provider,
    output_dir=out,
    output_filename="features.json",
)

# 4. Generate features:
generate_features_from_texts(
    root_folder=train_dir,
    discovered_features_path=out / "features.json",
    provider=provider,
    classes=['high', 'low'],
    output_dir=out / "train",
    merge_to_single_csv=True,
    merged_csv_name="train.csv",
)

generate_features_from_texts(
    root_folder=test_dir,
    discovered_features_path=out / "features.json",
    provider=provider,
    classes=['high', 'low'],
    output_dir=out / "test",
    merge_to_single_csv=True,
    merged_csv_name="test.csv",
)

train_feat = pd.read_csv(out / "train" / "train.csv")
test_feat = pd.read_csv(out / "test" / "test.csv")
pd.concat([train_feat, test_feat]).to_csv(out / "all_features.csv", index=False)

# 5. Evaluate:
# Baseline: TF-IDF
baseline = Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression())])
baseline.fit(train['text'], train['label'])
baseline_acc = accuracy_score(test['label'], baseline.predict(test['text']))

# LLM features + Decision Tree
X_train = pd.get_dummies(train_feat.drop(columns=['File', 'Class', 'raw_llm_output']))
X_test = pd.get_dummies(test_feat.drop(columns=['File', 'Class', 'raw_llm_output']))
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

tree = DecisionTreeClassifier(max_depth=3)
tree.fit(X_train, train_feat['Class'])
llm_acc = accuracy_score(test_feat['Class'], tree.predict(X_test))

# Results
print("\n" + "="*40)
print(f"TF-IDF Accuracy:     {baseline_acc:.1%}")
print(f"LLM Features Acc:    {llm_acc:.1%}")
print("="*40)

# Top features
imp = pd.DataFrame({'feature': X_train.columns, 'imp': tree.feature_importances_}).sort_values('imp', ascending=False)
print("\nTop 3 LLM-discovered features:")
print(imp.head(3).to_string(index=False))


[notice] A new release of pip is available: 25.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Train: 28 | Test: 12
Features saved to outputs/features.json


low: 100%|██████████| 6/6 [00:16<00:00,  2.78s/file]



TF-IDF Accuracy:     41.7%
LLM Features Acc:    58.3%

Top 3 LLM-discovered features:
                           feature      imp
personal_growth_significant_growth 0.481481
                magic_use_no_magic 0.272865
    foreign_setting_not observable 0.245654
